# X Full-Archive Keyword Scraper

Notebook-only scraper for `GET /2/tweets/search/all`.

References:
- https://docs.x.com/x-api/posts/search-all-posts
- https://docs.x.com/x-api/posts/search/integrate/operators
- https://docs.x.com/x-api/posts/search/quickstart/full-archive-search

Set `X_BEARER_TOKEN` in your environment before running the scrape cells. The notebook saves a flattened CSV under `data/raw/twitter/`, and trimmed JSONL API pages plus a manifest under `data/raw/twitter/JSON/`.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import random
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "keywords").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing docs/keywords and data directories.")


ROOT_DIR = find_repo_root()
KEYWORD_ROOT_DIR = ROOT_DIR / "docs" / "keywords"
KEYWORD_DIR = KEYWORD_ROOT_DIR / "by_language"
OUTPUT_DIR = ROOT_DIR / "data" / "raw" / "twitter"
JSON_OUTPUT_DIR = OUTPUT_DIR / "JSON"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root : {ROOT_DIR}")
print(f"Keywords  : {KEYWORD_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"JSON dir  : {JSON_OUTPUT_DIR}")

Repo root : /Users/gelo-p1/Documents/GitHub/HealthPH-Plus
Keywords  : /Users/gelo-p1/Documents/GitHub/HealthPH-Plus/docs/keywords/by_language
Output dir: /Users/gelo-p1/Documents/GitHub/HealthPH-Plus/data/raw/twitter
JSON dir  : /Users/gelo-p1/Documents/GitHub/HealthPH-Plus/data/raw/twitter/JSON


## Configuration

`START_TIME` and `END_TIME` accept UTC ISO format (`YYYY-MM-DDTHH:mm:ssZ`) or date-only values (`YYYY-MM-DD`). Date-only start values expand to `00:00:00Z`; date-only end values expand to `23:59:59Z`. Leave either as `None` for a broad/default API window.

In [ ]:
# Required auth. Do not hard-code tokens in the notebook.
BEARER_TOKEN = os.getenv("X_BEARER_TOKEN")

def normalize_x_api_time(value: str | None, *, bound: str) -> str | None:
    if value is None:
        return None
    value = value.strip()
    if not value:
        return None
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        suffix = "T00:00:00Z" if bound == "start" else "T23:59:59Z"
        return f"{value}{suffix}"
    try:
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError as exc:
        raise ValueError(f"{bound}_time must be YYYY-MM-DD or an ISO timestamp with timezone: {value}") from exc
    if parsed.tzinfo is None:
        raise ValueError(f"{bound}_time must include a timezone: {value}")
    return parsed.astimezone(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


# Full-archive search settings. Leave START_TIME/END_TIME as None unless intentionally bounding the scrape.
START_TIME = "2025-06-01"
END_TIME = "2025-12-31"
START_TIME = normalize_x_api_time(START_TIME, bound="start")
END_TIME = normalize_x_api_time(END_TIME, bound="end")
if START_TIME and END_TIME and START_TIME >= END_TIME:
    raise ValueError(f"START_TIME must be earlier than END_TIME: {START_TIME} >= {END_TIME}")
MAX_RESULTS = 150
SORT_ORDER = "recency"

# Keep requests public-safe and minimal. `public_metrics` is needed for `like_count`.
DEFAULT_FIELD_PROFILE = "public"

# Keep this small for a test run, then raise it when the output looks correct.
MAX_PAGES_PER_QUERY = 2
REQUEST_SLEEP_SECONDS = 2.0

# Self-serve full-archive search query limit. Raise to 4096 only if your account supports it.
QUERY_CHAR_LIMIT = 1024

# Optional filters appended to every query. Examples: ["-is:retweet"], ["place_country:PH"].
EXTRA_QUERY_FILTERS: list[str] = []

# Set each language to True/False depending on which keyword files you want to use.
USE_CEBUANO_KEYWORDS = True
USE_FILIPINO_KEYWORDS = True
USE_HILIGAYNON_KEYWORDS = True
USE_ILOCANO_KEYWORDS = True
USE_ENGLISH_KEYWORDS = False

# Shuffle keyword order before batching so repeated runs do not always start from the same terms.
SHUFFLE_KEYWORDS = True
KEYWORD_SHUFFLE_SEED = int(datetime.now(timezone.utc).timestamp()) if SHUFFLE_KEYWORDS else None

KEYWORD_LANGUAGE_FLAGS = {
    "cebuano": USE_CEBUANO_KEYWORDS,
    "filipino": USE_FILIPINO_KEYWORDS,
    "hiligaynon": USE_HILIGAYNON_KEYWORDS,
    "ilocano": USE_ILOCANO_KEYWORDS,
    "english": USE_ENGLISH_KEYWORDS,
}


def keyword_language_for_file(path: Path) -> str:
    relative_parts = [part.casefold() for part in path.relative_to(KEYWORD_DIR).parts]
    for language in KEYWORD_LANGUAGE_FLAGS:
        if language in relative_parts or path.name.casefold().startswith(f"{language}_"):
            return language
    return path.stem.casefold().removesuffix("_keywords")


# Load language keyword CSVs only, including nested language folders if present.
ALL_KEYWORD_FILES = sorted({path.resolve() for path in KEYWORD_DIR.rglob("*.csv")})
KEYWORD_FILES = [
    path for path in ALL_KEYWORD_FILES
    if KEYWORD_LANGUAGE_FLAGS.get(keyword_language_for_file(path), False)
]
KEYWORD_COLUMN_BY_FILE: dict[str, str] = {}

RUN_ID = datetime.now(timezone.utc).strftime("x_full_archive_%Y%m%dT%H%M%SZ")
CSV_OUTPUT_FILE = OUTPUT_DIR / f"{RUN_ID}.csv"
JSONL_OUTPUT_FILE = JSON_OUTPUT_DIR / f"{RUN_ID}.jsonl"
MANIFEST_OUTPUT_FILE = JSON_OUTPUT_DIR / f"{RUN_ID}_manifest.json"

print(f"Run ID        : {RUN_ID}")
print(f"CSV           : {CSV_OUTPUT_FILE}")
print(f"JSONL         : {JSONL_OUTPUT_FILE}")
print(f"Manifest      : {MANIFEST_OUTPUT_FILE}")
print(f"Token set     : {bool(BEARER_TOKEN)}")
print(f"Keyword flags : {KEYWORD_LANGUAGE_FLAGS}")
print(f"Shuffle keys  : {SHUFFLE_KEYWORDS}")
print(f"Shuffle seed  : {KEYWORD_SHUFFLE_SEED}")
print(f"Start time    : {START_TIME}")
print(f"End time      : {END_TIME}")
print(f"Field profile : {DEFAULT_FIELD_PROFILE}")

Run ID        : x_full_archive_20260729T022031Z
CSV           : /Users/gelo-p1/Documents/GitHub/HealthPH-Plus/data/raw/twitter/x_full_archive_20260729T022031Z.csv
JSONL         : /Users/gelo-p1/Documents/GitHub/HealthPH-Plus/data/raw/twitter/JSON/x_full_archive_20260729T022031Z.jsonl
Manifest      : /Users/gelo-p1/Documents/GitHub/HealthPH-Plus/data/raw/twitter/JSON/x_full_archive_20260729T022031Z_manifest.json
Token set     : True
Keyword flags : {'cebuano': True, 'filipino': True, 'hiligaynon': True, 'ilocano': True, 'english': False}
Shuffle keys  : True
Shuffle seed  : 1785291631
Start time    : 2025-06-01T00:00:00Z
End time      : 2025-12-31T23:59:59Z
Field profile : public


In [3]:
def normalize_keyword(value: Any) -> str | None:
    if value is None or pd.isna(value):
        return None
    keyword = str(value).strip().strip('"').strip("'")
    keyword = re.sub(r"\s+", " ", keyword)
    return keyword or None


def split_keyword_cell(value: Any) -> list[str]:
    keyword = normalize_keyword(value)
    if not keyword:
        return []
    try:
        return [part.strip() for part in next(csv.reader([keyword])) if part.strip()]
    except csv.Error:
        return [part.strip() for part in keyword.split(",") if part.strip()]


def load_first_column_keywords(path: Path) -> list[str]:
    keywords: list[str] = []
    header_names = {"keyword", "keywords", "term", "terms", "symptom", "symptoms"}
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        for row_number, row in enumerate(csv.reader(f)):
            if not row:
                continue
            first_cell = normalize_keyword(row[0])
            if row_number == 0 and first_cell and first_cell.casefold() in header_names:
                continue
            keywords.extend(split_keyword_cell(row[0]))
    return keywords


def load_keywords_from_file(path: Path, column: str | None = None) -> list[str]:
    if column is None:
        return load_first_column_keywords(path)

    df = pd.read_csv(path, dtype=str).fillna("")
    if df.empty:
        return []
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in {path}. Found: {df.columns.tolist()}")

    keywords: list[str] = []
    for value in df[column].tolist():
        keywords.extend(split_keyword_cell(value))
    return keywords


def load_keywords_from_files(paths: list[Path]) -> list[str]:
    seen: set[str] = set()
    merged: list[str] = []
    for path in paths:
        column = KEYWORD_COLUMN_BY_FILE.get(path.name)
        for keyword in load_keywords_from_file(path, column=column):
            key = keyword.casefold()
            if key in seen:
                continue
            seen.add(key)
            merged.append(keyword)
    return merged


def quote_query_term(keyword: str) -> str:
    keyword = keyword.replace("\\", "\\\\").replace('"', '\\"')
    if re.search(r"\s", keyword) or any(ch in keyword for ch in ["-", "/", "'", ","]):
        return f'"{keyword}"'
    return keyword


def with_extra_filters(query: str) -> str:
    filters = " ".join(part.strip() for part in EXTRA_QUERY_FILTERS if part.strip())
    return f"({query}) {filters}" if filters else query


def batch_keywords_for_queries(keywords: list[str], char_limit: int = QUERY_CHAR_LIMIT) -> list[str]:
    batches: list[str] = []
    current_terms: list[str] = []

    def render(terms: list[str]) -> str:
        return with_extra_filters(" OR ".join(terms))

    for keyword in keywords:
        term = quote_query_term(keyword)
        if len(with_extra_filters(term)) > char_limit:
            print(f"Skipping overlong keyword for query limit: {keyword[:80]}")
            continue
        candidate_terms = [*current_terms, term]
        if current_terms and len(render(candidate_terms)) > char_limit:
            batches.append(render(current_terms))
            current_terms = [term]
        else:
            current_terms = candidate_terms

    if current_terms:
        batches.append(render(current_terms))
    return batches


keywords = load_keywords_from_files(KEYWORD_FILES)
if SHUFFLE_KEYWORDS:
    random.Random(KEYWORD_SHUFFLE_SEED).shuffle(keywords)

query_batches = batch_keywords_for_queries(keywords)

print(f"Keyword files : {len(KEYWORD_FILES)}")
for path in KEYWORD_FILES:
    print(f"- {path.relative_to(ROOT_DIR)}")
print(f"Keywords      : {len(keywords)}")
print(f"Shuffled      : {SHUFFLE_KEYWORDS}")
print(f"Shuffle seed  : {KEYWORD_SHUFFLE_SEED}")
print(f"Query batches : {len(query_batches)}")
print("First query:")
print(query_batches[0] if query_batches else "<none>")

Keyword files : 4
- docs/keywords/by_language/cebuano_keywords.csv
- docs/keywords/by_language/filipino_keywords.csv
- docs/keywords/by_language/hiligaynon_keywords.csv
- docs/keywords/by_language/ilocano_keywords.csv
Keywords      : 405
Shuffled      : True
Shuffle seed  : 1785291631
Query batches : 9
First query:
"barado ang ilong" OR "nakapuy ti bagi" OR "indi makatilaw" OR panghihina OR hilantan OR nagkahuyang OR "daw may hilanat" OR pagkapagod OR "hindi nawawala ang ubo" OR "aguy-uyek" OR "walang gana sa pagkain" OR nagaalibadbad OR "uy-uyek" OR "nagahagok ang ginhawa" OR "green ti plema" OR "awan turog" OR "narigat ti panagturog" OR "nagaalab ang dughan" OR "nadulaan gana" OR "malapot nga plema" OR kapoy OR "puno ang ulo" OR "gab-i nga singot" OR ginahilanat OR "budlay magginhawa" OR "indi gana magkaon" OR "nahilo gamay" OR "dakes ti rikna" OR "pananakit ng katawan" OR "saan a makaanggo" OR "nagahuni ang ginhawa" OR nagapangurog OR "katubig-tubig" OR "kinakapos sa paghinga" OR "u

In [4]:
SEARCH_ALL_URL = "https://api.x.com/2/tweets/search/all"
PUBLIC_TWEET_FIELDS = ["created_at", "id", "text", "public_metrics", "geo"]
PLACE_FIELDS = ["id", "name", "full_name", "country", "country_code", "place_type"]
EXPANSIONS = ["geo.place_id"]


def comma(values: list[str]) -> str:
    return ",".join(values)


def search_params(query: str, next_token: str | None = None, profile: str = DEFAULT_FIELD_PROFILE) -> dict[str, str | int]:
    params: dict[str, str | int] = {
        "query": query,
        "max_results": MAX_RESULTS,
        "sort_order": SORT_ORDER,
        "tweet.fields": comma(PUBLIC_TWEET_FIELDS),
        "place.fields": comma(PLACE_FIELDS),
        "expansions": comma(EXPANSIONS),
    }
    if START_TIME:
        params["start_time"] = START_TIME
    if END_TIME:
        params["end_time"] = END_TIME
    if next_token:
        params["pagination_token"] = next_token
    return params


def summarize_api_errors(errors: list[dict[str, Any]] | None, limit: int = 10) -> dict[str, Any]:
    errors = errors or []
    title_counts = Counter(str(error.get("title", "<missing>")) for error in errors)
    field_counts = Counter(str(error.get("field", "")) for error in errors if error.get("field"))
    first_errors = [
        {key: error.get(key) for key in ["title", "detail", "field", "type"] if error.get(key)}
        for error in errors[:3]
    ]
    return {
        "count": len(errors),
        "titles": dict(title_counts.most_common(limit)),
        "fields": dict(field_counts.most_common(limit)),
        "first_errors": first_errors,
    }


def get_existing_ids(csv_paths: list[Path]) -> set[str]:
    ids: set[str] = set()
    for path in csv_paths:
        if not path.exists() or path.stat().st_size == 0:
            continue
        try:
            df = pd.read_csv(path, usecols=["id"], dtype=str)
        except ValueError:
            continue
        ids.update(df["id"].dropna().astype(str).str.replace("tweet-", "", regex=False).str.strip())
    return {post_id for post_id in ids if post_id}


existing_post_ids = get_existing_ids(sorted(OUTPUT_DIR.glob("*.csv")))
print(f"Existing Twitter/X IDs found for dedupe: {len(existing_post_ids)}")

Existing Twitter/X IDs found for dedupe: 27804


In [5]:
PREPROCESSED_TWITTER_COLUMNS = [
    "created_at",
    "id",
    "text",
    "like_count",
    "place_id",
    "place_name",
    "place_full_name",
    "place_country",
    "place_country_code",
    "place_type",
]


def places_by_id(payload: dict[str, Any]) -> dict[str, dict[str, Any]]:
    places = (payload.get("includes") or {}).get("places") or []
    return {str(place.get("id", "")).strip(): place for place in places if place.get("id")}


def flatten_post(post: dict[str, Any], places: dict[str, dict[str, Any]] | None = None) -> dict[str, Any]:
    public_metrics = post.get("public_metrics") or {}
    geo = post.get("geo") or {}
    place_id = str(geo.get("place_id", "")).strip()
    place = (places or {}).get(place_id, {})

    return {
        "created_at": post.get("created_at"),
        "id": str(post.get("id", "")).strip(),
        "text": post.get("text"),
        "like_count": public_metrics.get("like_count"),
        "place_id": place_id or None,
        "place_name": place.get("name"),
        "place_full_name": place.get("full_name"),
        "place_country": place.get("country"),
        "place_country_code": place.get("country_code"),
        "place_type": place.get("place_type"),
    }


def append_jsonl(path: Path, item: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False, sort_keys=True) + "\n")


def trimmed_response(payload: dict[str, Any]) -> dict[str, Any]:
    places = places_by_id(payload)
    return {
        "data": [flatten_post(post, places) for post in payload.get("data") or []],
        "meta": payload.get("meta") or {},
    }


def append_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows).reindex(columns=PREPROCESSED_TWITTER_COLUMNS)
    file_exists = path.exists() and path.stat().st_size > 0
    df.to_csv(path, mode="a", index=False, header=not file_exists, encoding="utf-8-sig")


In [6]:
def request_search_page(session: requests.Session, query: str, next_token: str | None, profile: str) -> tuple[dict[str, Any], str, int]:
    params = search_params(query=query, next_token=next_token, profile=profile)
    response = session.get(SEARCH_ALL_URL, params=params, timeout=60)
    request_count = 1

    if response.status_code == 429:
        reset_at = response.headers.get("x-rate-limit-reset")
        sleep_seconds = REQUEST_SLEEP_SECONDS
        if reset_at and reset_at.isdigit():
            sleep_seconds = max(int(reset_at) - int(time.time()) + 2, REQUEST_SLEEP_SECONDS)
        print(f"Rate limited. Sleeping {sleep_seconds:.0f} seconds before retry.")
        time.sleep(sleep_seconds)
        response = session.get(SEARCH_ALL_URL, params=params, timeout=60)
        request_count += 1

    if not response.ok:
        raise requests.HTTPError(f"HTTP {response.status_code}: {response.text[:1000]}", response=response)

    return response.json(), DEFAULT_FIELD_PROFILE, request_count


def scrape_query(session: requests.Session, query: str, query_index: int, seen_ids: set[str], manifest: dict[str, Any]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    next_token: str | None = None
    profile = DEFAULT_FIELD_PROFILE

    for page_number in range(1, MAX_PAGES_PER_QUERY + 1):
        payload, profile, request_count = request_search_page(session, query=query, next_token=next_token, profile=profile)
        manifest["request_count"] += request_count
        manifest["field_profiles_used"].append(profile)
        manifest["final_field_profile_by_query"][str(query_index)] = profile

        api_errors = payload.get("errors") or []
        if api_errors:
            manifest["api_error_summaries"].append({
                "query_index": query_index,
                "page": page_number,
                "field_profile": profile,
                "summary": summarize_api_errors(api_errors),
            })

        append_jsonl(JSONL_OUTPUT_FILE, {
            "run_id": RUN_ID,
            "query_index": query_index,
            "page": page_number,
            "field_profile": profile,
            "query": query,
            "start_time": START_TIME,
            "end_time": END_TIME,
            "response": trimmed_response(payload),
        })

        page_rows = []
        data = payload.get("data") or []
        places = places_by_id(payload)
        for post in data:
            post_id = str(post.get("id", "")).strip()
            if not post_id or post_id in seen_ids:
                manifest["duplicate_count"] += 1
                continue
            seen_ids.add(post_id)
            page_rows.append(flatten_post(post, places))

        append_csv(CSV_OUTPUT_FILE, page_rows)
        rows.extend(page_rows)

        meta = payload.get("meta") or {}
        result_count = meta.get("result_count", len(data))
        error_count = len(api_errors)
        print(
            f"Query {query_index}, page {page_number}: "
            f"result_count={result_count}, data_len={len(data)}, "
            f"error_count={error_count}, field_profile={profile}, new_rows={len(page_rows)}"
        )

        if not data and api_errors:
            print(f"API returned no data with errors: {summarize_api_errors(api_errors)}")

        next_token = meta.get("next_token")
        if not next_token:
            break
        time.sleep(REQUEST_SLEEP_SECONDS)

    return rows

## Run Scrape

For a dry run, keep `query_batches[:1]` and `MAX_PAGES_PER_QUERY = 1`. Increase both after checking the generated CSV and JSONL.

In [7]:
if not BEARER_TOKEN:
    raise RuntimeError("Set X_BEARER_TOKEN in your environment before running the scraper.")
if not query_batches:
    raise RuntimeError("No query batches were built from keyword files.")

manifest: dict[str, Any] = {
    "run_id": RUN_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "endpoint": SEARCH_ALL_URL,
    "keyword_dir": str(KEYWORD_DIR.relative_to(ROOT_DIR)),
    "keyword_files": [str(path.relative_to(ROOT_DIR)) for path in KEYWORD_FILES],
    "keyword_count": len(keywords),
    "keyword_language_flags": KEYWORD_LANGUAGE_FLAGS,
    "shuffle_keywords": SHUFFLE_KEYWORDS,
    "keyword_shuffle_seed": KEYWORD_SHUFFLE_SEED,
    "query_batches": query_batches,
    "query_batch_count": len(query_batches),
    "query_char_limit": QUERY_CHAR_LIMIT,
    "extra_query_filters": EXTRA_QUERY_FILTERS,
    "start_time": START_TIME,
    "end_time": END_TIME,
    "max_results": MAX_RESULTS,
    "max_pages_per_query": MAX_PAGES_PER_QUERY,
    "sort_order": SORT_ORDER,
    "default_field_profile": DEFAULT_FIELD_PROFILE,
    "csv_output_file": str(CSV_OUTPUT_FILE.relative_to(ROOT_DIR)),
    "jsonl_output_file": str(JSONL_OUTPUT_FILE.relative_to(ROOT_DIR)),
    "manifest_output_file": str(MANIFEST_OUTPUT_FILE.relative_to(ROOT_DIR)),
    "request_count": 0,
    "row_count": 0,
    "duplicate_count": 0,
    "field_profiles_used": [],
    "final_field_profile_by_query": {},
    "api_error_summaries": [],
    "errors": [],
}

headers = {"Authorization": f"Bearer {BEARER_TOKEN}", "User-Agent": "HealthPHPlus-XFullArchiveScraper/1.0"}
seen_ids = set(existing_post_ids)
all_rows: list[dict[str, Any]] = []

with requests.Session() as session:
    session.headers.update(headers)
    for query_index, query in enumerate(query_batches, start=1):
        try:
            all_rows.extend(scrape_query(session, query=query, query_index=query_index, seen_ids=seen_ids, manifest=manifest))
        except Exception as exc:
            error = {"query_index": query_index, "query": query, "error": repr(exc)}
            manifest["errors"].append(error)
            print(f"Error on query {query_index}: {exc}")
        time.sleep(REQUEST_SLEEP_SECONDS)

manifest["row_count"] = len(all_rows)
manifest["completed_at"] = datetime.now(timezone.utc).isoformat()
manifest["field_profiles_used"] = sorted(set(manifest["field_profiles_used"]))

MANIFEST_OUTPUT_FILE.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, sort_keys=True), encoding="utf-8")

print(f"Rows written       : {manifest['row_count']}")
print(f"Duplicates skipped : {manifest['duplicate_count']}")
print(f"Requests made      : {manifest['request_count']}")
print(f"API error pages    : {len(manifest['api_error_summaries'])}")
print(f"CSV                : {CSV_OUTPUT_FILE}")
print(f"JSONL              : {JSONL_OUTPUT_FILE}")
print(f"Manifest           : {MANIFEST_OUTPUT_FILE}")

Query 1, page 1: result_count=328, data_len=328, error_count=0, field_profile=public, new_rows=8
Query 1, page 2: result_count=320, data_len=320, error_count=0, field_profile=public, new_rows=7
Query 1, page 3: result_count=301, data_len=301, error_count=0, field_profile=public, new_rows=9
Query 1, page 4: result_count=332, data_len=332, error_count=0, field_profile=public, new_rows=82
Query 1, page 5: result_count=338, data_len=338, error_count=0, field_profile=public, new_rows=328
Query 1, page 6: result_count=331, data_len=331, error_count=0, field_profile=public, new_rows=323
Query 1, page 7: result_count=337, data_len=337, error_count=0, field_profile=public, new_rows=160
Query 1, page 8: result_count=336, data_len=336, error_count=0, field_profile=public, new_rows=44
Query 1, page 9: result_count=324, data_len=324, error_count=0, field_profile=public, new_rows=14
Query 1, page 10: result_count=332, data_len=332, error_count=0, field_profile=public, new_rows=12
Query 2, page 1: re

## Output Checks

In [8]:
if CSV_OUTPUT_FILE.exists() and CSV_OUTPUT_FILE.stat().st_size > 0:
    preview_df = pd.read_csv(CSV_OUTPUT_FILE, dtype=str)
    print(preview_df.shape)
    display(preview_df.head())
else:
    print("No CSV rows were written for this run.")

if JSONL_OUTPUT_FILE.exists() and JSONL_OUTPUT_FILE.stat().st_size > 0:
    with JSONL_OUTPUT_FILE.open("r", encoding="utf-8") as f:
        first_page = json.loads(next(f))
    first_response = first_page.get("response", {})
    print(first_page.keys())
    print("meta:", first_response.get("meta", {}))
    print("has_data:", bool(first_response.get("data")), "data_len:", len(first_response.get("data") or []))
    if first_response.get("data"):
        print("data keys:", list(first_response["data"][0].keys()))
else:
    print("No JSONL pages were written for this run.")

(14862, 10)


,created_at,id,text,like_count,place_id,place_name,place_full_name,place_country,place_country_code,place_type
0,2025-12-31T19:11:22.000Z,2006443078740226395,Kay kapoy na kaayo ning isotret. Sooo drying!!,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-12-30T17:34:46.000Z,2006056382186099001,Kapoy,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-12-30T13:30:43.000Z,2005994961398108568,@vnlluvv next time wag ka sumagot kung di mo g...,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-12-30T08:13:49.000Z,2005915214596956208,i'm soooo kapoy brah,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-12-30T07:53:43.000Z,2005910153577824521,kapoy ug lakaw uy intawon tapos ako pay maghik...,0,NaN,NaN,NaN,NaN,NaN,NaN


dict_keys(['end_time', 'field_profile', 'page', 'query', 'query_index', 'response', 'run_id', 'start_time'])
meta: {'newest_id': '2006513109591273739', 'next_token': 'b26v89c19zqg8o3jue8jkdw2uh98tp1kyqwzewuak4oe5', 'oldest_id': '2005497498962665641', 'result_count': 328}
has_data: True data_len: 328
data keys: ['created_at', 'id', 'like_count', 'place_country', 'place_country_code', 'place_full_name', 'place_id', 'place_name', 'place_type', 'text']
